# 02. Experimental Execution: Vector Baseline vs. Graph + PPR

**Course**: CS 328 — Introduction to Data Science  
**Project**: *Does Graph-Based Retrieval Beat Vector Search on Multi-Hop Questions?*  

This notebook executes and visualizes:
1. **Arm 1: Dense Vector Baseline** (SentenceTransformers + FAISS IndexFlatIP)
2. **Arm 2: Graph + PPR** (LLM OpenIE Entity Graph + Random Hyperplane LSH + Personalized PageRank)
3. **Arm 3: No-Walk Control (Ablation)** (Personalized PageRank with $\alpha = 0$, mass stays on seeds)
4. **Arm 4: Sensitivity Analysis** ($\alpha = 0.85$)
5. **Downstream LLM Reader Evaluation** (Short canonical answers across all 4 context arms)

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
from src import config
from src.data_loader import get_or_create_splits
from src.baseline_retriever import DenseBaselineRetriever
from src.graph_builder import GraphBuilder
from src.entity_linker import EntityLinker
from src.ppr_retriever import PPRRetriever

eval_split, _ = get_or_create_splits()
print(f"Loaded evaluation set with {len(eval_split.questions)} questions.")

2026-11-12 00:46:20,334 [INFO] Loading faiss.


2026-11-12 00:46:20,357 [INFO] Successfully loaded faiss.


/Users/hemcharanvarma/Desktop/ids_project/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1112 00:46:24.510000 11745 .venv/lib/python3.12/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


2026-11-12 00:46:25,711 [INFO] Found existing serialized splits. Loading directly...


2026-11-12 00:46:25,712 [INFO] Loading split from /Users/hemcharanvarma/Desktop/ids_project/data/2wikimultihopqa/eval_split.json...


2026-11-12 00:46:25,732 [INFO] Loading split from /Users/hemcharanvarma/Desktop/ids_project/data/2wikimultihopqa/tune_split.json...


Loaded evaluation set with 200 questions.


## 1. Dense Baseline Retrieval (FAISS IndexFlatIP)

In [2]:
dense_retriever = DenseBaselineRetriever()
dense_retriever.build_index(eval_split, cache_prefix="eval")
dense_results = dense_retriever.evaluate_split(eval_split)

r5_v = [res["recall@5"] for res in dense_results.values()]
r2_v = [res["recall@2"] for res in dense_results.values()]
print(f"Vector Baseline recall@5: {np.mean(r5_v)*100:.2f}%")
print(f"Vector Baseline recall@2: {np.mean(r2_v)*100:.2f}%")

2026-11-12 00:46:25,747 [INFO] Loading SentenceTransformer: sentence-transformers/all-MiniLM-L6-v2 on cpu...


2026-11-12 00:46:26,129 [INFO] HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2 "HTTP/1.1 200 OK"


2026-11-12 00:46:26,139 [INFO] Loading SentenceTransformer model from sentence-transformers/all-MiniLM-L6-v2.


Loading weights:   0%|                                 | 0/103 [00:00<?, ?it/s]

Loading weights: 100%|████████████████████| 103/103 [00:00<00:00, 13682.13it/s]

2026-11-12 00:46:26,535 [INFO] HTTP Request: GET https://huggingface.co/api/models/sentence-transformers/all-MiniLM-L6-v2/tree/main/additional_chat_templates?recursive=false&expand=false "HTTP/1.1 404 Not Found"


... (7 outputs left out)

Batches:  25%|████████▊                          | 1/4 [00:00<00:00,  7.67it/s]

Batches:  50%|█████████████████▌                 | 2/4 [00:00<00:00,  8.56it/s]

Batches: 100%|███████████████████████████████████| 4/4 [00:00<00:00, 12.04it/s]

Vector Baseline recall@5: 67.12%
Vector Baseline recall@2: 56.88%


## 2. Graph Construction & LSH Entity Linking

In [3]:
graph_builder = GraphBuilder()
G, e2p, p2e = graph_builder.build_graph(eval_split, cache_name="eval")
print(f"Knowledge Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges")

linker = EntityLinker(graph=G, embedder=dense_retriever.embedder)
synonyms_added = linker.augment_graph_with_synonyms(threshold=config.TAU_SYNONYM)
linker.precompute_query_entities([q.question for q in eval_split.questions], batch_size=10)
print(f"Augmented with {synonyms_added} intra-graph synonym edges.")

2026-11-12 00:46:27,490 [INFO] Loading cached graph from /Users/hemcharanvarma/Desktop/ids_project/cache/graphs/eval_graph.json...


2026-11-12 00:46:27,544 [INFO] Loaded graph with 7948 nodes and 10254 edges.


2026-11-12 00:46:27,549 [INFO] Embedding 7948 graph nodes for EntityLinker LSH...


Knowledge Graph: 7948 nodes, 10254 edges


2026-11-12 00:46:33,532 [INFO] Hyperplane LSH indexed 7948 entities across 16 bands (r=8, l=128).


2026-11-12 00:46:34,789 [INFO] Evaluating 4107968 candidate pairs from LSH buckets...


2026-11-12 00:46:49,335 [INFO] Discovered 2158 intra-graph synonym edges with cosine >= 0.80


2026-11-12 00:46:49,683 [INFO] Added 1695 new synonym edges to graph G.


2026-11-12 00:46:49,683 [INFO] All query entities already cached.


Augmented with 1695 intra-graph synonym edges.


## 3. Personalized PageRank Retrieval (Primary vs. No-Walk Control)

In [4]:
ppr_retriever = PPRRetriever(
    graph=G,
    entity_to_passages=e2p,
    passage_to_entities=p2e,
    entity_linker=linker,
    dense_retriever=dense_retriever,
    alpha=config.ALPHA_PRIMARY,
)

# Arm 2: Graph + PPR (alpha=0.5)
graph_results, teleport_cache = ppr_retriever.evaluate_split(eval_split, dense_results=dense_results, alpha=0.5)
# Arm 3: No-Walk Control (alpha=0.0)
nowalk_results, _ = ppr_retriever.evaluate_split(eval_split, dense_results=dense_results, alpha=0.0, teleport_cache=teleport_cache)

r5_g = [res["recall@5"] for res in graph_results.values()]
r5_nw = [res["recall@5"] for res in nowalk_results.values()]
print(f"Graph + PPR (alpha=0.5) recall@5: {np.mean(r5_g)*100:.2f}%")
print(f"No-Walk Control (alpha=0.0) recall@5: {np.mean(r5_nw)*100:.2f}%")

2026-11-12 00:46:49,690 [INFO] Initialized PPRRetriever (alpha=0.5, tol=1e-12, nodes=7948)


2026-11-12 00:46:49,691 [INFO] Evaluating 200 questions on Graph + PPR (alpha=0.5)...


2026-11-12 00:46:55,415 [INFO] Evaluation finished for alpha=0.5. Fallback triggered on 0/200 (0.0%) queries.


2026-11-12 00:46:55,416 [INFO] Evaluating 200 questions on Graph + PPR (No-Walk Control (alpha=0))...


2026-11-12 00:46:55,478 [INFO] Evaluation finished for No-Walk Control (alpha=0). Fallback triggered on 0/200 (0.0%) queries.


Graph + PPR (alpha=0.5) recall@5: 93.75%
No-Walk Control (alpha=0.0) recall@5: 71.75%
